# 07 · Publishing Packages: `twine`, `uv publish`, `poetry publish`

**Module 08 – Modules, Packaging & Dependency Management** · Estimated time: 60 min

> ❗ Not in either source. You'll take the wheels and sdists from lesson 06 and **publish** them, first to a **local package index running inside this notebook** (so every command really runs), then learn the exact steps for **TestPyPI** and **PyPI**.

### 🎯 What you will learn
- How package indexes work (the "simple" API)
- PyPI vs TestPyPI vs private indexes; accounts, 2FA, **API tokens**, `.pypirc`
- **twine**: `check`, `upload`, repositories, `--skip-existing`
- **`uv publish`** and **`poetry publish`**
- ✅ **Trusted Publishing** from GitHub Actions (no secrets), attestations
- Versioning & release workflow, yanking, security (typosquatting, tokens)

In [1]:
import os, sys, tempfile, pathlib, shutil, subprocess, socket, time, urllib.request
LAB = pathlib.Path(tempfile.mkdtemp(prefix="publish-lab-"))
os.environ.update({
    "LAB": str(LAB), "PY": sys.executable,
    "PATH": os.path.dirname(sys.executable) + os.pathsep + os.environ["PATH"],
    "UV_CACHE_DIR": str(LAB / ".uv-cache"), "UV_PYTHON": sys.executable, "UV_PYTHON_DOWNLOADS": "never",
    "UV_NO_PROGRESS": "1", "NO_COLOR": "1", "PIP_DISABLE_PIP_VERSION_CHECK": "1",
    "POETRY_CACHE_DIR": str(LAB / ".poetry-cache"), "POETRY_CONFIG_DIR": str(LAB / ".poetry-config"),
    "POETRY_DATA_DIR": str(LAB / ".poetry-data"), "POETRY_VIRTUALENVS_IN_PROJECT": "true",
    "GIT_CONFIG_GLOBAL": os.devnull,
})
for var in ("FORCE_COLOR", "CLICOLOR_FORCE"):
    os.environ.pop(var, None)
print(LAB)

/var/folders/96/f6xk04z91_gdystl3qfqv33r0000gn/T/publish-lab-63udvj2x


## 1. How package indexes work

An index is just a web server following the **Simple Repository API** (PEP 503/691):
- `GET /simple/` → list of projects
- `GET /simple/<project>/` → links to every file (wheel/sdist) with hashes
- `POST /legacy/` (on PyPI: `https://upload.pypi.org/legacy/`) → upload endpoint

| Index | Upload URL | Install URL |
|---|---|---|
| **PyPI** | `https://upload.pypi.org/legacy/` | `https://pypi.org/simple/` |
| **TestPyPI** (practice) | `https://test.pypi.org/legacy/` | `https://test.pypi.org/simple/` |
| Private | e.g. devpi, pypiserver, GitLab/GitHub Packages, AWS CodeArtifact, Artifactory, Nexus | their URLs |

## 2. Start a local package index (for this notebook only)

We use **pypiserver** (`pip install pypiserver`), started in the background with authentication disabled:

In [2]:
PKG_DIR = LAB / "index-packages"
PKG_DIR.mkdir()
with socket.socket() as s:                 # find a free port
    s.bind(("127.0.0.1", 0))
    PORT = s.getsockname()[1]

server = subprocess.Popen(
    [sys.executable, "-m", "pypiserver", "run", "-p", str(PORT), "-a", ".", "-P", ".",
     "--disable-fallback", "--overwrite", str(PKG_DIR)],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
INDEX = f"http://127.0.0.1:{PORT}"
for _ in range(50):                         # wait until it answers
    try:
        urllib.request.urlopen(INDEX + "/simple/", timeout=1)
        break
    except OSError:
        time.sleep(0.2)
os.environ["INDEX"] = INDEX
print("Local index running at", INDEX)

Local index running at http://127.0.0.1:64629


## 3. A package to publish

In [3]:
%%bash
cd "$LAB"
uv init --lib --author-from none --vcs none greetkit-demo >/dev/null 2>&1
cd greetkit-demo
cat > src/greetkit_demo/__init__.py <<'EOF'
"""A tiny demo package."""
def greet(name: str) -> str:
    return f"Hello, {name}! 👋"
EOF
printf '# greetkit-demo\n\nA tiny demo package used to learn publishing.\n' > README.md
uv version 0.1.0 --no-sync >/dev/null 2>&1
uv build 2>&1 | tail -2

Successfully built dist/greetkit_demo-0.1.0.tar.gz
Successfully built dist/greetkit_demo-0.1.0-py3-n

one-any.whl


## 4. `twine check`: validate before uploading

In [4]:
%%bash
cd "$LAB/greetkit-demo"
twine check dist/*

Checking dist/greetkit_demo-0.1.0-py3-none-any.whl: 

PASSED


Checking dist/greetkit_demo-0.1.0.tar.gz: 

PASSED


It verifies the metadata and that the README will render on PyPI (Markdown needs `readme = "README.md"`, which sets the content type).

## 5. `twine upload`

In [5]:
%%bash
cd "$LAB/greetkit-demo"
twine upload --repository-url "$INDEX/" -u demo -p demo --non-interactive --disable-progress-bar dist/* 2>&1 | grep -v "^$"
echo "--- the index now lists:"
curl -s "$INDEX/simple/greetkit-demo/" | grep -o 'greetkit_demo[^"<#]*' | sort -u

Uploading distributions to http://127.0.0.1:64629/
Uploading greetkit_demo-0.1.0-py3-none-any.whl
Up

loading greetkit_demo-0.1.0.tar.gz


--- the index now lists:


greetkit_demo-0.1.0-py3-none-any.whl
greetkit_demo-0.1.0.tar.gz


### twine against the real PyPI/TestPyPI

1. Create accounts on **https://test.pypi.org** and **https://pypi.org** (separate accounts), and enable **2FA** (required by PyPI).
2. Create an **API token** (*Account settings → API tokens*). Scope it to a single project after the first upload.
3. Upload. The username is literally `__token__`, and the password is the token:

```bash
python -m twine upload --repository testpypi dist/*        # try TestPyPI first!
python -m pip install --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ greetkit-demo
python -m twine upload dist/*                              # then the real PyPI
```

`~/.pypirc` (keep it private: `chmod 600 ~/.pypirc`):
```ini
[distutils]
index-servers =
    pypi
    testpypi

[pypi]
username = __token__
password = pypi-AgEIcHlwaS5vcmc...

[testpypi]
repository = https://test.pypi.org/legacy/
username = __token__
password = pypi-AgENdGVzdC5weXBp...
```
or environment variables (best in CI): `TWINE_USERNAME=__token__`, `TWINE_PASSWORD=pypi-...`, `TWINE_REPOSITORY_URL=...`.

| twine option | Meaning |
|---|---|
| `-r / --repository NAME` | a section from `.pypirc` (`testpypi`) |
| `--repository-url URL` | an explicit upload URL |
| `--skip-existing` | don't fail when a file already exists (re-running CI) |
| `--attestations` | upload PEP 740 attestations alongside the files |
| `--non-interactive` | never prompt (CI) |
| `--verbose` | debug upload errors |

> ⚠️ **Old way:** `python setup.py upload` / `setup.py register` (insecure, removed). Also `twine register` is no longer needed for PyPI: projects are created on first upload.

## 6. Install from the index

In [6]:
%%bash
cd "$LAB"
"$PY" -m venv consumer
consumer/bin/python -m pip install -q --index-url "$INDEX/simple/" greetkit-demo
consumer/bin/python -c "from greetkit_demo import greet; print(greet('PyPI learner'))"

Hello, PyPI learner! 👋


## 7. `uv publish`

Bump the version (PyPI never lets you re-upload the same filename) and publish with uv:

In [7]:
%%bash
cd "$LAB/greetkit-demo"
uv version --bump patch --no-sync
rm -rf dist && uv build 2>&1 | tail -1
uv publish --publish-url "$INDEX/" --username demo --password demo 2>&1 | tail -3
curl -s "$INDEX/simple/greetkit-demo/" | grep -oE 'greetkit_demo-[0-9]+\.[0-9]+\.[0-9]+' | sort -u

Using CPython 3.15.0 interpreter at: /private/tmp/claude-501/-Users-gaurvendrasingh-Desktop-python/f

4ffaf20-c7ab-4c1c-ae64-d5055060b56d/scratchpad/venv315/bin/python


Resolved 1 package in 6ms


greetkit-demo 0.1.0 => 0.1.1


Successfully built dist/greetkit_demo-0.1.1-py3-none-any.whl


Uploading greetkit_demo-0.1.1-py3-none-any.whl (1.5KiB)
Hashing greetkit_demo-0.1.1.tar.gz (893B)
Up

loading greetkit_demo-0.1.1.tar.gz (893B)


greetkit_demo-0.1.0
greetkit_demo-0.1.1


```bash
uv publish                              # to PyPI; reads UV_PUBLISH_TOKEN or uses trusted publishing in CI
uv publish --token pypi-AgEI...         # explicit token
uv publish --index testpypi             # a named index from pyproject.toml (needs publish-url):
```
```toml
[[tool.uv.index]]
name = "testpypi"
url = "https://test.pypi.org/simple/"
publish-url = "https://test.pypi.org/legacy/"
explicit = true
```
Other options: `--dry-run`, `--check-url URL` (skip files that already exist), `--trusted-publishing always|never|automatic`, `--no-attestations`.

## 8. `poetry publish`

In [8]:
%%bash
cd "$LAB"
poetry new --author "Demo <demo@example.com>" poetry-greet-demo >/dev/null 2>&1
cd poetry-greet-demo
printf 'def hi() -> str:\n    return "hi from poetry"\n' > src/poetry_greet_demo/__init__.py
poetry config repositories.local "$INDEX/"
poetry publish --build -r local -u demo -p demo 2>&1 | tail -4
curl -s "$INDEX/simple/" | grep -o 'poetry-greet-demo\|greetkit-demo' | sort -u

 - Uploading poetry_greet_demo-0.1.0.tar.gz 0%
 - Uploading poetry_greet_demo-0.1.0.tar.gz 100%
 - U

ploading poetry_greet_demo-0.1.0-py3-none-any.whl 0%
 - Uploading poetry_greet_demo-0.1.0-py3-none-a

ny.whl 100%


greetkit-demo
poetry-greet-demo


```bash
poetry config pypi-token.pypi pypi-AgEI...                 # store the token once (keyring)
poetry publish --build                                      # build + upload to PyPI
poetry config repositories.testpypi https://test.pypi.org/legacy/
poetry config pypi-token.testpypi pypi-AgEN...
poetry publish -r testpypi --dry-run                        # rehearse
poetry publish --skip-existing                              # idempotent CI runs
```

## 9. ✅ Trusted Publishing (recommended for CI)

Long-lived API tokens can leak. **Trusted Publishing** lets GitHub Actions (or GitLab, Google Cloud, ActiveState) prove its identity to PyPI with a short-lived **OIDC** token: **no secrets stored anywhere**.

1. On PyPI: *Your project → Publishing → Add a trusted publisher* (owner, repo, workflow file name, environment name). For a brand-new project use a *pending publisher*.
2. Workflow `.github/workflows/release.yml`:

```yaml
name: release
on:
  push:
    tags: ["v*"]                 # publish when you push a tag like v1.2.0

jobs:
  build:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v5
      - uses: astral-sh/setup-uv@v7
      - run: uv build
      - uses: actions/upload-artifact@v4
        with: { name: dist, path: dist/ }

  publish:
    needs: build
    runs-on: ubuntu-latest
    environment: pypi            # must match the trusted-publisher settings
    permissions:
      id-token: write            # ← REQUIRED for trusted publishing
    steps:
      - uses: actions/download-artifact@v4
        with: { name: dist, path: dist/ }
      - uses: pypa/gh-action-pypi-publish@release/v1      # or: uv publish
```
The PyPA action also creates **PEP 740 attestations** (signed provenance linking each file to the workflow run that built it), which PyPI shows on the project page.

> 💡 Pin actions to the latest major versions (or full commit SHAs for maximum supply-chain safety).

## 10. Versioning and the release workflow

- Follow **PEP 440**; most projects use **semantic versioning** `MAJOR.MINOR.PATCH`:
  - MAJOR: breaking changes · MINOR: new features · PATCH: bug fixes
  - pre-releases `2.0.0a1`, `2.0.0b1`, `2.0.0rc1`; installers ignore them unless asked (`--pre`)
- Bump with `uv version --bump`, `poetry version`, `hatch version`, or derive from Git tags (`hatch-vcs`, `setuptools-scm`, `poetry-dynamic-versioning`)
- Keep a `CHANGELOG.md` ("Keep a Changelog" format)

**Release checklist**
1. Tests & linters green on CI
2. Bump version, update changelog, commit
3. `git tag v1.2.0 && git push --tags` → CI builds and publishes (trusted publishing)
4. Verify: `pip install yourpkg==1.2.0` in a fresh venv
5. Create a GitHub Release with the changelog

## 11. Rules & gotchas on PyPI

- A **filename can never be re-uploaded**, even after deletion. Bump the version (`1.2.0.post1` for packaging-only fixes).
- **Yank**, don't delete, broken releases (*Manage → Yank*): installers skip yanked versions unless pinned exactly.
- Project names are normalised: `My_Pkg`, `my-pkg`, `my.pkg` are the **same** project.
- **Security:** enable 2FA; use project-scoped tokens or trusted publishing; never commit tokens (`.pypirc` belongs in your home folder only); beware **typosquatting** when installing (`reqeusts` ≠ `requests`); consider `uv audit` / `pip-audit` for your dependencies.
- **Private code**: never publish to the public PyPI. Use a private index and configure it as the *default* (`--default-index`, Poetry `primary` source) to avoid **dependency confusion** attacks.

## 12. Shut down the local index

In [9]:
server.terminate()
server.wait(timeout=10)
print("local index stopped; uploaded files:", sorted(p.name for p in PKG_DIR.iterdir()))

local index stopped; uploaded files: ['greetkit_demo-0.1.0-py3-none-any.whl', 'greetkit_demo-0.1.0.tar.gz', 'greetkit_demo-0.1.1-py3-none-any.whl', 'greetkit_demo-0.1.1.tar.gz', 'poetry_greet_demo-0.1.0-py3-none-any.whl', 'poetry_greet_demo-0.1.0.tar.gz']


## 📝 Summary
- Check with `twine check`, rehearse on **TestPyPI**, then publish to **PyPI**.
- `twine upload` (`__token__` + API token, `.pypirc`, `--skip-existing`), `uv publish`, `poetry publish --build`.
- ✅ Prefer **Trusted Publishing** from CI: no stored secrets, plus attestations.
- Versions are immutable on PyPI; yank instead of delete; protect accounts with 2FA.

## 🏋️ Practice
1. Create a TestPyPI account, make a uniquely named package (e.g. `yourname-hello`), and publish it with `uv publish --index testpypi`.
2. Install it from TestPyPI into a fresh venv and import it.
3. Set up a GitHub repository with the release workflow above, configure a pending trusted publisher on TestPyPI, and publish by pushing a tag.
4. Run your own pypiserver (`pypi-server run -p 8080 ./packages`) and point `uv add --index` at it.